In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import yfinance as yf
import statsmodels.api as sm
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
end = dt.date.today()
start = end - dt.timedelta(365*6)
df = yf.download(["^NSEI", "^NSEBANK"], start=start, end=end, auto_adjust=True)

[*********************100%***********************]  2 of 2 completed


In [55]:
# Cleaning Data
df.columns = df.columns.droplevel(1)
df.columns = df.columns.str.lower()
df.drop(columns=["high",'low','open','volume'], inplace=True)
df.columns = ['bank','nse']

# Creating Returns
df['bank_returns'] = df['bank'].pct_change()
df['nse_returns'] = df['nse'].pct_change()
df.dropna(inplace=True)

# Splitting Data
split = int(len(df)*0.8)
train = df[:split]
test = df[split:]
y_train = train['bank_returns']
y_test = test['bank_returns']
x_train = train['nse_returns']
x_test = test['nse_returns']

# Model Training
x_train = sm.add_constant(x_train)
x_test = sm.add_constant(x_test)
model = sm.OLS(y_train,x_train).fit()
pred = model.predict(x_test)
error = y_test - pred

# Summary
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           bank_returns   R-squared:                       0.772
Model:                            OLS   Adj. R-squared:                  0.772
Method:                 Least Squares   F-statistic:                     4009.
Date:                Fri, 20 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:25:38   Log-Likelihood:                 4178.6
No. Observations:                1184   AIC:                            -8353.
Df Residuals:                    1182   BIC:                            -8343.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0002      0.000     -1.095      